# SGD、Momentum 与 Adam

**面试回答：**SGD 用当前梯度更新，Momentum 累积一致方向，Adam 用一二阶矩自适应缩放；需比较同一目标和训练预算。

## 真实案例

用一个简化的商家服务时长回归目标演示不同优化器在狭长损失面的更新。

In [1]:
import numpy as np  # 导入 NumPy 手写优化器。
print('教学目标：w0 接近 3，w1 接近 1，w1 方向曲率更大。')  # 说明业务化简目标。
def gradient(w):  # 定义简化损失的梯度。
    return np.array([2*(w[0]-3),20*(w[1]-1)])  # 返回两个参数方向的梯度。
def loss(w):  # 定义服务时长误差代理损失。
    return (w[0]-3)**2+10*(w[1]-1)**2  # 返回狭长二次损失。
print('初始损失=',loss(np.array([0.,0.])))  # 输出初始状态。

教学目标：w0 接近 3，w1 接近 1，w1 方向曲率更大。
初始损失= 19.0


## Baseline / 基线

固定步长 SGD 是对照。

In [2]:
sgd=np.array([0.,0.])  # 初始化 SGD 参数。
for step in range(20):  # 运行固定步长 SGD。
    sgd-=.04*gradient(sgd)  # 按当前梯度更新参数。
print('SGD 参数/损失=',np.round(sgd,3),round(float(loss(sgd)),4))  # 输出基线结果。

SGD 参数/损失= [2.434 1.   ] 0.3204


In [3]:
momentum=np.array([0.,0.])  # 初始化 Momentum 参数。
velocity=np.zeros(2)  # 初始化速度向量。
adam=np.array([0.,0.])  # 初始化 Adam 参数。
m=np.zeros(2)  # 初始化 Adam 一阶矩。
v=np.zeros(2)  # 初始化 Adam 二阶矩。
for step in range(1,21):  # 同预算运行两种优化器。
    velocity=.8*velocity+gradient(momentum)  # 累积 Momentum 速度。
    momentum-=.04*velocity  # 更新 Momentum 参数。
    g=gradient(adam)  # 计算 Adam 当前梯度。
    m=.9*m+.1*g  # 更新一阶矩。
    v=.999*v+.001*g*g  # 更新二阶矩。
    m_hat=m/(1-.9**step)  # 执行一阶偏差修正。
    v_hat=v/(1-.999**step)  # 执行二阶偏差修正。
    adam-=.12*m_hat/(np.sqrt(v_hat)+1e-8)  # 按自适应步长更新参数。
print('Momentum 参数/损失=',np.round(momentum,3),round(float(loss(momentum)),4))  # 输出 Momentum 结果。
print('Adam 参数/损失=',np.round(adam,3),round(float(loss(adam)),4))  # 输出 Adam 结果。

Momentum 参数/损失= [2.809 0.946] 0.0657
Adam 参数/损失= [2.212 1.256] 1.2755


## 结果解读

Momentum 平滑震荡，Adam 缩小高曲率方向步长；小玩具任务不能证明线上泛化优势。

In [4]:
print('优化器 | 20步损失')  # 输出对比表头。
print('SGD',round(float(loss(sgd)),4))  # 输出 SGD。
print('Momentum',round(float(loss(momentum)),4))  # 输出 Momentum。
print('Adam',round(float(loss(adam)),4))  # 输出 Adam。
print('生产差距：需固定 batch、token、学习率搜索、梯度裁剪和验证回放。')  # 说明边界。

优化器 | 20步损失
SGD 0.3204
Momentum 0.0657
Adam 1.2755
生产差距：需固定 batch、token、学习率搜索、梯度裁剪和验证回放。


## 失败案例与修复

忽略 Adam 偏差修正会使早期矩估计偏小；修复是使用 m_hat/v_hat。

In [5]:
uncorrected=m/(np.sqrt(v)+1e-8)  # 故意计算未偏差修正的更新比例。
corrected=m_hat/(np.sqrt(v_hat)+1e-8)  # 计算正确更新比例。
print('失败未修正比例=',np.round(uncorrected,3))  # 输出失败中间量。
print('修复偏差修正比例=',np.round(corrected,3))  # 输出修复中间量。
print('AdamW 的 weight decay 仍需单独处理。')  # 补充工程边界。

失败未修正比例= [-4.783  1.299]
修复偏差修正比例= [-0.766  0.208]
AdamW 的 weight decay 仍需单独处理。


In [6]:
assert loss(sgd)<loss(np.zeros(2))  # 保护 SGD 有学习。
assert loss(momentum)<loss(np.zeros(2))  # 保护 Momentum 有学习。
assert loss(adam)<loss(np.zeros(2))  # 保护 Adam 有学习。
assert not np.allclose(uncorrected,corrected)  # 保护偏差修正差异。